# DOCX → Azota — làm lại từ đầu

**Không cần Google Drive.** Code nằm `/content/docx-to-azota`, zip tải về máy.

Log Kaggle `OCR HOÀN TẤT` / 5 trang PDF (~57s/trang) **không** phải pipeline này. Đề `.docx` chạy **Phần A**: ô A5 in `EXTRACT HOÀN TẤT` cùng khuôn (giây/phút + thống kê), xong trong ~0.2s.

**Xóa runtime cũ:** Runtime → Disconnect and delete runtime. Upload **chỉ** notebook này. Không dán cell từ Untitled1 / chat.

1. **Phần A** (clone → zip) — CPU, **không** UniMERNet. Đủ file Azota. **Đủ 6 ô A1–A6.**
2. **Phần B** — chỉ khi cần `$latex$` từ MathType.

Cấm: `unimernet[full]` / `pip install tokenizers` / `transformers==4.42.4`.
Đừng bấm Stop lúc clone. Không Run all.


## Phần A — extract Azota


### A1. GPU (không bắt buộc cho extract)


In [ ]:
!nvidia-smi -L || echo CPU


### A2. Clone code **một lần** (không Drive)


In [ ]:
import shutil, sys, subprocess
from pathlib import Path
for p in ("/content/repo", "/content/docx-to-azota", "/content/_repo_azota", "/content/refurbished-marketplace"):
    shutil.rmtree(p, ignore_errors=True)
REPO = "https://github.com/phuchoang2603/refurbished-marketplace.git"
BRANCH = "cursor/docx-to-azota-pipeline-4d56"
subprocess.check_call([
    "git", "clone", "-b", BRANCH, "--depth", "1", "--single-branch",
    "--filter=blob:none", "--sparse", REPO, "/content/repo",
])
subprocess.check_call(["git", "-C", "/content/repo", "sparse-checkout", "set", "tools/docx-to-azota"])
src = Path("/content/repo/tools/docx-to-azota/convert.py")
if not src.exists():
    raise SystemExit("clone chưa đủ file — chạy lại ô này, đừng bấm Stop")
shutil.copytree(src.parent, "/content/docx-to-azota")
ROOT = Path("/content/docx-to-azota")
sys.path.insert(0, str(ROOT))
print("OK", (ROOT / "convert.py").exists())


### A3. Import converter (không load UniMERNet)


In [ ]:
!pip -q install pillow
import sys
from pathlib import Path
ROOT = Path("/content/docx-to-azota")
sys.path.insert(0, str(ROOT))
from convert import convert_docx
from eval_timer import StepTimer
timer = StepTimer()
OUT = "/content/azota_out"
print("import OK")


### A4. Upload `.docx`


In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    DOCX = "/content/" + next(iter(uploaded))
else:
    DOCX = "/content/docx-to-azota/samples/de-vat-li-lan-3.docx"
print(DOCX)


### A5. Extract — **bắt buộc**. Kỳ vọng mathml 69, mathtype 16, img 8.


In [ ]:
import time
from pathlib import Path
from eval_timer import print_extract_complete

t0 = time.perf_counter()
with timer.step("Bước 1", "OOXML"):
    man = convert_docx(DOCX, OUT)
elapsed = time.perf_counter() - t0
print_extract_complete(DOCX, elapsed, man["counts"])
print("\n--- 25 dòng markup ---")
print("\n".join(Path(OUT, "markup.txt").read_text(encoding="utf-8").splitlines()[:25]))


### A6. Tải zip — **có thể dừng tại đây**


In [ ]:
from google.colab import files
!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
print("xong phần A")


## Phần B — tùy chọn: MathType → LaTeX

Chỉ chạy nếu Azota cần `$latex$` thay `[!m:$mathtype_N$]`. T4. Không OCR hình. Không Google Drive.


### B1. ImageMagick


In [ ]:
!apt-get -qq install -y imagemagick libmagickwand-dev
!pip -q install Wand huggingface_hub
print("ImageMagick OK")


### B2. UniMERNet `--no-deps` (tự gắn `/content/docx-to-azota`)

Restart session mất `sys.path`. Ô này tìm `/content/docx-to-azota` hoặc clone lại — **không** hỏi Drive.

Thành công khi in `unimernet OK ...`. Dòng pip `ERROR: ... requires transformers==4.42.4` **bỏ qua** (cố ý giữ Colab 5.15).


In [ ]:
import sys, shutil, subprocess
from pathlib import Path

def _complete(p: Path) -> bool:
    return p.is_dir() and (p / "install_colab.py").is_file() and (p / "convert.py").is_file()

ROOT = next((p for p in (
    Path("/content/docx-to-azota"),
    Path("/content/repo/tools/docx-to-azota"),
    Path("/content/_repo_azota/tools/docx-to-azota"),
) if _complete(p)), None)

if ROOT is None:
    REPO = "https://github.com/phuchoang2603/refurbished-marketplace.git"
    BRANCH = "cursor/docx-to-azota-pipeline-4d56"
    STAGING = Path("/content/repo")
    shutil.rmtree(STAGING, ignore_errors=True)
    subprocess.check_call([
        "git", "clone", "-b", BRANCH, "--depth", "1", "--single-branch",
        "--filter=blob:none", "--sparse", REPO, str(STAGING),
    ])
    subprocess.check_call(["git", "-C", str(STAGING), "sparse-checkout", "set", "tools/docx-to-azota"])
    src = STAGING / "tools/docx-to-azota"
    if not (src / "install_colab.py").is_file():
        raise SystemExit("clone chưa đủ file — chạy lại ô này, đừng bấm Stop")
    shutil.rmtree("/content/docx-to-azota", ignore_errors=True)
    shutil.copytree(src, "/content/docx-to-azota")
    ROOT = Path("/content/docx-to-azota")

sys.path.insert(0, str(ROOT))
OUT = "/content/azota_out"
print("ROOT", ROOT)
print("install_colab.py", (ROOT / "install_colab.py").exists())

from install_colab import allow_wmf_in_imagemagick, install_unimernet_colab
allow_wmf_in_imagemagick()
install_unimernet_colab()


### B3. Raster WMF + nhận dạng + gắn `$latex$`


In [ ]:
import sys
from pathlib import Path
from google.colab import files

ROOT = Path("/content/docx-to-azota")
sys.path.insert(0, str(ROOT))
OUT = "/content/azota_out"

from convert import convert_docx
from eval_timer import StepTimer
from colab_opt import prepare_unimernet_checkpoint, free_cuda, vision_jobs_from_manifest, inject_latex_into_markup
from vision import rasterize_formula_image, load_unimernet, unimernet_batch
from convert import apply_unimernet_latex

timer = StepTimer()

def find_docx() -> Path:
    top = sorted(Path("/content").glob("*.docx"), key=lambda p: p.stat().st_mtime, reverse=True)
    if top:
        return top[0]
    sample = ROOT / "samples" / "de-vat-li-lan-3.docx"
    if sample.is_file():
        return sample
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("Cần file .docx — upload đề rồi chạy lại ô này")
    return Path("/content") / next(iter(uploaded))

DOCX = find_docx()
print("DOCX", DOCX)
with timer.step("Bước 1", "OOXML"):
    man = convert_docx(DOCX, OUT)
print(man["counts"])

png_dir = Path(OUT) / "sidecar_png"
png_dir.mkdir(exist_ok=True)
jobs = []
with timer.step("Bước 2", "raster WMF"):
    for aid, src in vision_jobs_from_manifest(man, OUT, kinds=("mathtype",)):
        dest = png_dir / f"{aid}.png"
        got = rasterize_formula_image(src, dest, dpi=200)
        if got:
            jobs.append((aid, got))
print(len(jobs), "ảnh công thức")

with timer.step("Bước 3-load", "tiny"):
    cfg = prepare_unimernet_checkpoint("tiny", "/content/models")
    model, vis, device = load_unimernet(cfg_path=cfg, fp16=True)
print("device", device)

with timer.step("Bước 3", "batch"):
    preds = unimernet_batch(model, vis, device, jobs, batch_size=8)
apply_unimernet_latex(man, preds, Path(OUT))
p = Path(OUT) / "markup.txt"
text = inject_latex_into_markup(p.read_text(encoding="utf-8"), preds)
p.write_text(text, encoding="utf-8")
timer.print_summary()
for k, v in list(preds.items())[:5]:
    print(k, "→", v[:80])
free_cuda(model, vis)

!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
